# Raw data cleaning

This notebook cleans and validates the merged raw dataset created in
`01_merge_raw_datasets.ipynb`.

The objectives are limited to:

1. importing the merged raw dataset;
2. preserving an untouched copy;
3. checking column names and data types;
4. identifying source-specific structural missingness;
5. identifying special missing-value codes;
6. checking categorical values and numeric ranges;
7. applying documented technical cleaning;
8. removing the incompatible `Routine` variable;
9. saving a cleaned but substantively untransformed dataset.

No scales, composite variables, categories, or analytical variables are
constructed in this notebook.


## 1. Import packages and the merged raw dataset

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")


In [2]:
DATA_FOLDER = Path(
    r"C:\Users\JonatanPl\OneDrive - JDC\Desktop\Yoni\HighTechProtest - Sampling Wave 2\Python\excel"
)

INPUT_PATH = DATA_FOLDER / "merged_raw_dataset.xlsx"
OUTPUT_PATH = DATA_FOLDER / "clean_raw_dataset.xlsx"
LOG_PATH = DATA_FOLDER / "cleaning_log.xlsx"
INVENTORY_PATH = DATA_FOLDER / "variable_inventory_raw.xlsx"

print(f"Input file exists: {INPUT_PATH.exists()}")


Input file exists: True


In [3]:
df_raw = pd.read_excel(INPUT_PATH)

print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")


Rows: 860
Columns: 109


## 2. Preserve an untouched copy

All cleaning operations are performed on a copy. The imported dataframe
`df_raw` remains unchanged throughout the notebook.


In [4]:
df = df_raw.copy()

assert df.shape == (860, 109)
assert df["ResponseId"].is_unique
assert df["ResponseId"].notna().all()


## 3. Initialize the cleaning log

In [5]:
cleaning_log = pd.DataFrame(
    columns=[
        "step",
        "variable",
        "issue",
        "action",
        "affected_cases",
        "justification"
    ]
)

cleaning_log


,step,variable,issue,action,affected_cases,justification


## 4. Inspect and standardize column names

Column names are checked for leading or trailing spaces, internal spaces, and
duplicates. Only technical name standardization is performed.


In [6]:
column_audit = pd.DataFrame({
    "column": df.columns,
    "leading_or_trailing_space": [
        column != column.strip()
        for column in df.columns
    ],
    "contains_space": [
        " " in column
        for column in df.columns
    ],
    "duplicate_name": df.columns.duplicated()
})

column_audit.loc[
    column_audit[
        [
            "leading_or_trailing_space",
            "contains_space",
            "duplicate_name"
        ]
    ].any(axis=1)
]


,column,leading_or_trailing_space,contains_space,duplicate_name
5,Duration (in seconds),False,True,False


In [7]:
df.columns = df.columns.str.strip()

assert not df.columns.duplicated().any()


### Rename the duration variable

The Qualtrics duration variable contains spaces and parentheses. It is renamed
to a simpler machine-readable name. Its values are not changed.


In [8]:
if "Duration (in seconds)" in df.columns:
    df = df.rename(
        columns={
            "Duration (in seconds)": "DurationSeconds"
        }
    )

    cleaning_log = pd.concat(
        [
            cleaning_log,
            pd.DataFrame([{
                "step": "Standardize column name",
                "variable": "Duration (in seconds)",
                "issue": "Column name contains spaces and parentheses",
                "action": "Renamed to DurationSeconds",
                "affected_cases": len(df),
                "justification": "Improves consistency and usability in Python"
            }])
        ],
        ignore_index=True
    )

assert "DurationSeconds" in df.columns
assert "Duration (in seconds)" not in df.columns


## 5. Create a variable inventory

In [9]:
variable_inventory = pd.DataFrame({
    "variable": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "nonmissing_n": df.notna().sum().values,
    "missing_n": df.isna().sum().values,
    "missing_pct": df.isna().mean().mul(100).values,
    "unique_n": [
        df[column].nunique(dropna=True)
        for column in df.columns
    ]
})

variable_inventory = variable_inventory.sort_values(
    ["missing_pct", "variable"],
    ascending=[False, True]
)

variable_inventory


,variable,dtype,nonmissing_n,missing_n,missing_pct,unique_n
101,Ethnicity_5_TEXT,object,7,853,99.186,6
37,EmploymentStatus_7_TEXT,object,18,842,97.907,17
99,Residence_8_TEXT,object,27,833,96.860,24
96,Party_11_TEXT,object,52,808,93.953,36
43,Occupation_4_TEXT,object,90,770,89.535,77
106,FollowUp_Email,object,133,727,84.535,133
107,Notes,object,300,560,65.116,189
105,FollowUp_Interview,float64,404,456,53.023,2
97,BirthYear,float64,758,102,11.860,53
40,Manager,float64,794,66,7.674,2


In [10]:
variable_inventory.to_excel(
    INVENTORY_PATH,
    index=False
)


## 6. Identify source-specific structural missingness

The following variables were asked only in the Meta questionnaire. Missing
values among Panel respondents are expected by design.


In [11]:
meta_only_variables = [
    "Ethnicity_5_TEXT",
    "FollowUp_Email",
    "FollowUp_Interview"
]

structural_missingness = pd.DataFrame({
    "variable": meta_only_variables,
    "meta_nonmissing": [
        df.loc[df["data_source"] == "meta", variable].notna().sum()
        for variable in meta_only_variables
    ],
    "panel_nonmissing": [
        df.loc[df["data_source"] == "panel", variable].notna().sum()
        for variable in meta_only_variables
    ],
    "panel_missing": [
        df.loc[df["data_source"] == "panel", variable].isna().sum()
        for variable in meta_only_variables
    ]
})

structural_missingness


,variable,meta_nonmissing,panel_nonmissing,panel_missing
0,Ethnicity_5_TEXT,7,0,454
1,FollowUp_Email,133,0,454
2,FollowUp_Interview,404,0,454


## 7. Inspect imported data types

In [12]:
dtype_summary = (
    df.dtypes
    .astype(str)
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="number_of_variables")
)

dtype_summary


,dtype,number_of_variables
0,int64,53
1,float64,39
2,object,14
3,datetime64[ns],3


In [13]:
object_variables = df.select_dtypes(
    include=["object"]
).columns.tolist()

object_variables


['IPAddress',
 'ResponseId',
 'DistributionChannel',
 'UserLanguage',
 'EmploymentStatus_7_TEXT',
 'Occupation_4_TEXT',
 'Role_Description',
 'Attention_check2',
 'Party_11_TEXT',
 'Residence_8_TEXT',
 'Ethnicity_5_TEXT',
 'FollowUp_Email',
 'Notes',
 'data_source']

In [14]:
for variable in object_variables:
    unique_n = df[variable].nunique(dropna=True)

    if unique_n <= 20:
        print(f"\n{variable}")
        print(
            df[variable]
            .value_counts(dropna=False)
            .head(25)
        )



DistributionChannel
DistributionChannel
anonymous    860
Name: count, dtype: int64

UserLanguage
UserLanguage
HE    860
Name: count, dtype: int64

EmploymentStatus_7_TEXT
EmploymentStatus_7_TEXT
NaN                                                       842
עצמאית                                                      2
עצמאית ויזמת                                                1
בחופשת לידה                                                 1
א.כ.ע                                                       1
חופשת לידה                                                  1
גימלאי ויועץ עצמאי                                          1
משרת אם 8 שעות                                              1
סטודנט ועובד במשרה חלקית                                    1
חופשת לידה                                                  1
מעסיק, שותף בחברה                                           1
עבודה במשרה מלאה וסטודנטית                                  1
חופשת לידה (אבל לפני ואחרי עובדת במשרה מלאה)                

## 8. Search for special missing-value codes

In [15]:
text_missing_candidates = [
    "",
    " ",
    "NA",
    "N/A",
    "n/a",
    "null",
    "NULL",
    "None",
    "missing",
    "Missing"
]

candidate_counts = []

for variable in df.select_dtypes(include="object").columns:
    for candidate in text_missing_candidates:
        count = df[variable].eq(candidate).sum()

        if count > 0:
            candidate_counts.append({
                "variable": variable,
                "candidate": candidate,
                "count": count
            })

pd.DataFrame(candidate_counts)


""


In [16]:
numeric_missing_candidates = [
    -999,
    -99,
    -9,
    97,
    98,
    99,
    997,
    998,
    999,
    1752
]

numeric_candidate_counts = []

for variable in df.select_dtypes(include="number").columns:
    for candidate in numeric_missing_candidates:
        count = df[variable].eq(candidate).sum()

        if count > 0:
            numeric_candidate_counts.append({
                "variable": variable,
                "candidate": candidate,
                "count": count
            })

pd.DataFrame(numeric_candidate_counts)


,variable,candidate,count
0,DurationSeconds,997,1
1,DurationSeconds,999,1


## 9. Remove accidental surrounding whitespace

Leading and trailing spaces are removed from text variables. Empty strings are
converted to missing values. The number of changed text values is recorded.


In [17]:
text_columns = df.select_dtypes(include="object").columns

whitespace_changes = 0

for variable in text_columns:
    changed = df[variable].apply(
        lambda value: (
            isinstance(value, str)
            and value != value.strip()
        )
    )

    whitespace_changes += int(changed.sum())

    df[variable] = df[variable].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

df[text_columns] = df[text_columns].replace(
    r"^\s*$",
    np.nan,
    regex=True
)

print(f"Text values changed by whitespace removal: {whitespace_changes}")


Text values changed by whitespace removal: 226


In [18]:
cleaning_log = pd.concat(
    [
        cleaning_log,
        pd.DataFrame([{
            "step": "Remove surrounding whitespace",
            "variable": "All text variables",
            "issue": "Leading or trailing whitespace",
            "action": "Applied str.strip() and converted blank strings to missing",
            "affected_cases": whitespace_changes,
            "justification": "Prevents artificial duplicate text categories"
        }])
    ],
    ignore_index=True
)


## 10. Inspect categorical values

In [19]:
low_cardinality_variables = [
    variable
    for variable in df.columns
    if df[variable].nunique(dropna=True) <= 20
]

print(f"Low-cardinality variables: {len(low_cardinality_variables)}")


Low-cardinality variables: 93


In [20]:
for variable in low_cardinality_variables:
    print(f"\n{'=' * 70}")
    print(variable)

    display(
        pd.crosstab(
            df[variable],
            df["data_source"],
            margins=True,
            dropna=False
        )
    )



Status


data_source,meta,panel,All
Status,,,
0,406,454,860
All,406,454,860



Progress


data_source,meta,panel,All
Progress,,,
100,406,454,860
All,406,454,860



Finished


...

data_source,meta,panel,All
Salary,,,
1.000,0,6,6.000
2.000,0,5,5.000
3.000,7,15,22.000
4.000,23,94,117.000
5.000,58,160,218.000
6.000,315,170,485.000
NaN,3,4,NaN
All,406,454,860.000



FollowUp_Interview


data_source,meta,panel,All
FollowUp_Interview,,,
1.000,146,0,146.000
2.000,258,0,258.000
NaN,2,454,NaN
All,406,454,860.000



data_source


data_source,meta,panel,All
data_source,,,
meta,406,0,406
panel,0,454,454
All,406,454,860


## 11. Inspect numeric ranges

In [21]:
numeric_ranges = pd.DataFrame({
    "variable": df.select_dtypes(include="number").columns,
    "nonmissing_n": [
        df[variable].notna().sum()
        for variable in df.select_dtypes(include="number").columns
    ],
    "min": [
        df[variable].min()
        for variable in df.select_dtypes(include="number").columns
    ],
    "max": [
        df[variable].max()
        for variable in df.select_dtypes(include="number").columns
    ],
    "unique_n": [
        df[variable].nunique(dropna=True)
        for variable in df.select_dtypes(include="number").columns
    ]
})

numeric_ranges


,variable,nonmissing_n,min,max,unique_n
0,Status,860,0.000,0.000,1
1,Progress,860,100.000,100.000,1
2,DurationSeconds,860,270.000,270542.000,580
3,Finished,860,1.000,1.000,1
4,LocationLatitude,860,-37.816,52.520,85
5,LocationLongitude,860,-118.329,144.967,85
6,Consent,860,1.000,1.000,1
7,Gender,860,0.000,1.000,2
8,TechProtest_Dispute,859,1.000,4.000,4
9,TechProtest_Legit,860,1.000,5.000,5


## 12. Compare numeric ranges across sources

In [22]:
shared_numeric_variables = list(
    df.select_dtypes(include="number").columns
)

source_range_comparison = []

for variable in shared_numeric_variables:
    source_range_comparison.append({
        "variable": variable,
        "meta_min": df.loc[df["data_source"] == "meta", variable].min(),
        "meta_max": df.loc[df["data_source"] == "meta", variable].max(),
        "panel_min": df.loc[df["data_source"] == "panel", variable].min(),
        "panel_max": df.loc[df["data_source"] == "panel", variable].max()
    })

source_range_comparison = pd.DataFrame(
    source_range_comparison
)

source_range_comparison.loc[
    (source_range_comparison["meta_min"]
     != source_range_comparison["panel_min"])
    |
    (source_range_comparison["meta_max"]
     != source_range_comparison["panel_max"])
]


,variable,meta_min,meta_max,panel_min,panel_max
2,DurationSeconds,319.000,115604.000,270.000,270542.000
4,LocationLatitude,-37.816,52.520,14.505,52.382
5,LocationLongitude,-118.329,144.967,-118.261,136.906
28,Experience,1.000,60.000,1.000,55.000
31,Routine,1.000,4.000,1.000,6.000
49,V4,1.000,6.000,2.000,6.000
51,V14,2.000,6.000,1.000,6.000
85,BirthYear,1948.000,2001.000,1943.000,2000.000
87,Ethnicity,1.000,5.000,1.000,3.000
89,Education,1.000,7.000,1.000,8.000


## 13. Remove the `Routine` variable

The `Routine` variable uses incompatible numeric coding across the Meta and
Panel datasets. Because the response categories cannot be harmonized reliably
from the available raw files, the variable is removed from the cleaned dataset.

No respondent records are removed.


In [23]:
assert "Routine" in df.columns

df = df.drop(columns=["Routine"])

assert "Routine" not in df.columns

cleaning_log = pd.concat(
    [
        cleaning_log,
        pd.DataFrame([{
            "step": "Remove incompatible variable",
            "variable": "Routine",
            "issue": (
                "The Meta and Panel datasets use incompatible numeric coding "
                "for this variable"
            ),
            "action": "Dropped variable from the cleaned dataset",
            "affected_cases": len(df),
            "justification": (
                "The coding could not be harmonized reliably without confirmed "
                "response-label correspondence"
            )
        }])
    ],
    ignore_index=True
)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


Rows: 860
Columns: 108


## 14. Check exact duplicate rows

In [24]:
exact_duplicate_rows = df.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicate_rows}")


Exact duplicate rows: 0


## 15. Record unusually long survey durations

Long completion times are retained because both source datasets already passed
the predefined quality threshold. They may reflect respondents leaving the
survey window open.


In [25]:
df["DurationSeconds"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)


count      860.000
mean      2123.065
std      13164.618
min        270.000
50%        647.500
75%        891.250
90%       1439.400
95%       2397.800
99%      29163.880
max     270542.000
Name: DurationSeconds, dtype: float64

In [26]:
cleaning_log = pd.concat(
    [
        cleaning_log,
        pd.DataFrame([{
            "step": "Inspect survey duration",
            "variable": "DurationSeconds",
            "issue": "Several unusually long completion times",
            "action": "No change made",
            "affected_cases": pd.NA,
            "justification": (
                "Both source datasets already passed the predefined quality "
                "threshold; long durations may reflect an open browser session"
            )
        }])
    ],
    ignore_index=True
)


## 16. Validate the cleaned raw dataset

In [27]:
final_audit = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "meta_cases": df["data_source"].eq("meta").sum(),
    "panel_cases": df["data_source"].eq("panel").sum(),
    "unique_response_ids": df["ResponseId"].nunique(),
    "duplicate_response_ids": df["ResponseId"].duplicated().sum(),
    "missing_response_ids": df["ResponseId"].isna().sum(),
    "routine_present": "Routine" in df.columns
})

final_audit


rows                        860
columns                     108
meta_cases                  406
panel_cases                 454
unique_response_ids         860
duplicate_response_ids        0
missing_response_ids          0
routine_present           False
dtype: object

In [28]:
assert len(df) == 860
assert df.shape[1] == 108
assert df["data_source"].eq("meta").sum() == 406
assert df["data_source"].eq("panel").sum() == 454
assert df["ResponseId"].nunique() == 860
assert not df["ResponseId"].duplicated().any()
assert df["ResponseId"].notna().all()
assert "Routine" not in df.columns

print("All final validation checks passed.")


All final validation checks passed.


## 17. Save the cleaned data and cleaning log

The cleaned dataset retains all qualifying respondents. Only documented
technical cleaning operations have been applied.


In [29]:
df.to_excel(
    OUTPUT_PATH,
    index=False
)

cleaning_log.to_excel(
    LOG_PATH,
    index=False
)

print(f"Saved cleaned dataset to: {OUTPUT_PATH.resolve()}")
print(f"Saved cleaning log to: {LOG_PATH.resolve()}")


Saved cleaned dataset to: C:\Users\JonatanPl\OneDrive - JDC\Desktop\Yoni\HighTechProtest - Sampling Wave 2\Python\excel\clean_raw_dataset.xlsx
Saved cleaning log to: C:\Users\JonatanPl\OneDrive - JDC\Desktop\Yoni\HighTechProtest - Sampling Wave 2\Python\excel\cleaning_log.xlsx
